In [38]:
import torch
import transformer_lens
from sae_lens import SAE


In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [40]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.6.hook_resid_pre",
    device="cuda"
)

In [41]:
model = transformer_lens.HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)

C:\Users\Danii\AppData\Local\Temp\ipykernel_13152\2224763915.py:1: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = transformer_lens.HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [42]:
import json

with open('story.json') as f:
    data = json.load(f)
print(len(data))

20


In [43]:
alphas = [45]

ids = [4192, 20026, 4214, 8301, 19470, 4262, 13374, 15462, 12552]

service_ids = [
    # Articles
    8565, 9397, 10180,

    # Prepositions
    8438, 11490, 9857, 10992, 12086, 10268,
    10797, 11738, 11867, 8103, 838, 12368,

    # Conjunctions
    9439, 10255, 9622, 8207, 15190,
    19090, 2175, 20589,

    # Modal verbs and negation
    9991, 9438, 16376, 4009, 16358, 9618, 12046,


    4256, 9886, 1191, 9438, 3626, 8586, 5778
]

vector = sae.W_dec[20026].unsqueeze(0).unsqueeze(0)

In [44]:
vector.shape

torch.Size([1, 1, 768])

In [45]:
FUNCTION_WORDS = {
    # Articles and determiners
    "a", "an", "the", "this", "that", "these", "those",
    "each", "every", "either", "neither", "some", "any",
    "all", "both", "few", "many", "much", "several",

    # Pronouns
    "i", "me", "my", "mine", "myself",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself",
    "she", "her", "hers", "herself",
    "it", "its", "itself",
    "we", "us", "our", "ours", "ourselves",
    "they", "them", "their", "theirs", "themselves",
    "who", "whom", "whose", "which", "what",

    # Prepositions
    "of", "to", "in", "for", "on", "with", "at", "by",
    "from", "up", "about", "into", "over", "after",
    "beneath", "under", "above", "through", "during",
    "before", "between", "without", "within", "along",
    "across", "behind", "beyond", "toward", "towards",
    "against", "among", "around", "near",

    # Conjunctions
    "and", "but", "or", "nor", "so", "yet",
    "if", "because", "although", "though", "while",
    "when", "whenever", "where", "whereas", "whether",
    "unless", "until", "since", "than", "as",

    # Auxiliary verbs
    "am", "is", "are", "was", "were",
    "be", "been", "being",
    "have", "has", "had", "having",
    "do", "does", "did", "doing",
    "will", "would", "shall", "should",
    "can", "could", "may", "might", "must",

    # Particles and function-like adverbs
    "not", "no", "yes", "very", "too", "also",
    "only", "just", "even", "still", "already",
}


def build_token_sets(tokenizer):
    content_ids = []
    function_ids = []

    for token_id in range(len(tokenizer)):
        token_text = tokenizer.decode(
            [token_id],
            clean_up_tokenization_spaces=False
        )

        # Берём только GPT-2-токены, представляющие целое
        # слово с начальным пробелом.
        if not token_text.startswith(" "):
            continue

        word = token_text.strip().lower()

        # Исключаем числа, пунктуацию и BPE-фрагменты
        if not word.isalpha():
            continue

        if word in FUNCTION_WORDS:
            function_ids.append(token_id)

        elif len(word) >= 2:
            content_ids.append(token_id)

    return (
        torch.tensor(content_ids, dtype=torch.long),
        torch.tensor(function_ids, dtype=torch.long)
    )


content_ids, function_ids = build_token_sets(
    model.tokenizer
)

content_ids = content_ids.to(model.W_U.device)
function_ids = function_ids.to(model.W_U.device)

print("Content token count:", len(content_ids))
print("Function token count:", len(function_ids))

Content token count: 31682
Function token count: 357


In [46]:
device = model.W_U.device
dtype = model.W_U.dtype

W_U = model.W_U                              # [768, 50257]
W_dec = sae.W_dec.to(device=device, dtype=dtype)  # [24576, 768]

content_ids = content_ids.to(device)
function_ids = function_ids.to(device)

with torch.no_grad():
    content_direction = W_U[
        :, content_ids
    ].mean(dim=-1)                            # [768]

    function_direction = W_U[
        :, function_ids
    ].mean(dim=-1)                           # [768]

    content_contrast_direction = (
        content_direction - function_direction
    )                                        # [768]

    feature_content_scores = (
        W_dec @ content_contrast_direction
    ).detach()                               # [24576]

print(feature_content_scores.shape)
# torch.Size([24576])

torch.Size([24576])


In [47]:
lst_tensors = []

# Консервативный порог: gate срабатывает примерно на 10% позиций
CONTENT_THRESHOLD = 6.536377

gate_stats = {
    "calls": 0,
    "triggered": 0,
    "scores": []
}


def hook_steering(tensor, hook):
    """
    tensor: [batch, position, 768]
    feature_content_scores: [24576]
    vector: steering-вектор [768], [1, 768] или [1, 1, 768]
    """

    last_tensor = tensor[:, -1:, :]  # [B, 1, 768]

    # SAE-активации текущего токена
    sae_acts = sae.encode(last_tensor)  # [B, 1, 24576]

    scores = feature_content_scores.to(
        device=sae_acts.device,
        dtype=sae_acts.dtype
    ).reshape(1, 1, -1)

    # Оценка того, ожидается ли следующий смысловой токен
    current_content_score = (
        sae_acts * scores
    ).sum(
        dim=-1,
        keepdim=True
    )  # [B, 1, 1]

    gate = (
        current_content_score >= CONTENT_THRESHOLD
    )  # [B, 1, 1]

    steering_vector = vector.to(
        device=tensor.device,
        dtype=tensor.dtype
    ).reshape(1, 1, tensor.shape[-1])

    steered_last = (
        last_tensor
        + gate.to(tensor.dtype)
        * alpha
        * steering_vector
    )

    # Диагностика
    gate_stats["calls"] += tensor.shape[0]
    gate_stats["triggered"] += gate.sum().item()
    gate_stats["scores"].extend(
        current_content_score
        .detach()
        .flatten()
        .cpu()
        .tolist()
    )

    return torch.cat(
        [tensor[:, :-1, :], steered_last],
        dim=1
    )

In [48]:
idx = ids[2]

for idx_p, prompt in enumerate(data[:5]):
    for alpha in alphas:
        res = []
        print(f'prompt: {idx_p}')
        for i in range(1):
            torch.manual_seed(42 + i + idx_p)
            no_steering = model.generate(prompt,temperature=0.9, max_new_tokens=100,  top_p=0.9, top_k=50)

            model.add_hook(name="blocks.6.hook_resid_pre", hook = hook_steering, dir='fwd')
            torch.manual_seed(42 + i + idx_p)
            steering = model.generate(prompt,temperature=0.9, max_new_tokens=100,  top_p=0.9, top_k=50)
            model.reset_hooks()
            res.append({
                'prompt': prompt,
                'no_steering': no_steering,
                'steering': steering,
                'alpha': alpha,
                'seed': 42 + i + idx_p
            })
        with open(f'res_sae_service/3/{idx_p}_{alpha}.json', mode='w') as f:
            json.dump(res, f, indent=5)

prompt: 0


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

prompt: 1


  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

prompt: 4


  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [37]:
lst_tensors = torch.concat(lst_tensors, dim=0)

RuntimeError: torch.cat(): expected a non-empty list of Tensors

In [11]:
lst_tensors.shape

torch.Size([500, 1, 24576])

In [12]:
for row in lst_tensors:
    if (row[:,] > 1.0).any():
        print(row)


tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([[0., 0

In [13]:
assert lst_tensors.ndim == 3
assert lst_tensors.shape[1] == 1

# Убираем фиктивную размерность: [2000, 24576]
acts = lst_tensors[:, 0, :]

# По 10 максимальных значений для каждой строки
top_values, top_indices = torch.topk(
    acts,
    k=10,
    dim=-1
)

top_values = top_values.detach().cpu()
top_indices = top_indices.detach().cpu()

for row_idx, (indices, values) in enumerate(
    zip(top_indices, top_values)
):
    print(f"Строка {row_idx}:")

    for feature_id, activation in zip(indices.tolist(), values.tolist()):
        print(
            f"  feature={feature_id:5d}, "
            f"activation={activation:.6f}"
        )

Строка 0:
  feature= 5199, activation=11.964921
  feature=24480, activation=5.260795
  feature=13171, activation=2.699762
  feature= 9815, activation=2.472134
  feature= 4943, activation=2.312385
  feature=20983, activation=2.288651
  feature=16842, activation=2.030161
  feature=19316, activation=1.885830
  feature=19282, activation=1.820478
  feature=18170, activation=1.772934
Строка 1:
  feature=15285, activation=16.954939
  feature= 4244, activation=10.844790
  feature=20506, activation=5.650285
  feature= 7359, activation=5.217844
  feature= 5573, activation=3.988941
  feature= 1020, activation=3.369916
  feature=22663, activation=2.718735
  feature= 1225, activation=1.583035
  feature=  246, activation=1.205478
  feature=23734, activation=0.897145
Строка 2:
  feature= 6233, activation=17.584511
  feature= 4244, activation=8.964995
  feature=20506, activation=6.288312
  feature= 6203, activation=5.515327
  feature= 5573, activation=3.427040
  feature=  246, activation=1.838259
  fe

In [16]:
service_ids = [
    8565, 9397, 10180,
    8438, 11490, 9857, 10992, 12086, 10268,
    10797, 11738, 11867, 8103, 838, 12368,
    9439, 10255, 9622, 8207, 15190,
    19090, 2175, 20589,
    9991, 9438, 16376, 4009, 16358, 9618, 12046
]

service_tensor = torch.tensor(
    service_ids,
    device=device,
    dtype=torch.long
)

W_dec = sae.W_dec.to(
    device=device,
    dtype=model.W_U.dtype
)

W_U = model.W_U

# [number_of_service_features, vocabulary]
selected_logit_effects = (
    W_dec[service_tensor] @ W_U
)

# Нормализация внутри каждого feature
centered_effects = (
    selected_logit_effects
    - selected_logit_effects.mean(dim=-1, keepdim=True)
)

normalized_effects = (
    centered_effects
    / centered_effects.std(
        dim=-1,
        keepdim=True
    ).clamp_min(1e-8)
)

content_enrichment = normalized_effects[
    :, content_ids
].mean(dim=-1)

function_enrichment = normalized_effects[
    :, function_ids
].mean(dim=-1)

prediction_score = (
    content_enrichment
    - function_enrichment
)